#**From sequence representation → 2D & 3D molecular structures**

[Google Colab Notebook](https://colab.research.google.com/drive/1g-5FcVJQU-FORvjIBoA3AdXvh0xoEiGl#scrollTo=SYUoddM881zC)

[pyPept Repository](https://github.com/Boehringer-Ingelheim/pyPept/blob/master/examples/branched_peptide.py)

[Ochoa et al., pyPept (2023)]( https://doi.org/10.1186/s13321-023-00748-2)


##**pyPept Pipeline**

Peptides are short chains of amino acids that play key roles in biological systems and drug discovery.

In this project, we use `pyPept` to:

- Convert peptide sequences into molecular structures
- Transform HELM notation into BILN format
- Generate 2D molecular representations
- Build 3D conformers of peptides
- Visualize chemical structures computationally

##**pyPept: Strengths and Limitations**

Strengths
- Support for Specialized Peptide Format: pyPept is internally based on BILN, but it also can accept FASTA or HELM representations as input.
- Support for Non-Standard Amino Acids: Unlike many
basic peptide tools, pyPept can handle modified and non-natural amino acids, which is important in peptide drug design and biotechnology research.
- Advantage over AlphaFold for Modified Peptides: While AlphaFold can predict structures only for peptides composed of natural amino acids, pyPept can represent and model peptides containing non-natural residues directly. This makes pyPept especially useful for studying chemically modified peptides without requiring residue replacement and back-mutation steps.
- Generation of Molecular Structures: pyPept can generate 2D and 3D molecular representations of peptides, helping users visualize peptide structures and perform further computational analyses. The *Molecule class* takes this Sequence object and creates a molecule object with a sanitized **2D representation**. The *Conformer class* leverages distance geometry functionality to generate
a **3D conformer**.


Limitations
- Limited Input Format Support: pyPept primarily supports BILN representations and does not directly accept HELM and FASTA sequences as input.
- Sequence Length Limitation: Based on testing, the tool performs best with relatively short peptide sequences (approximately 20 amino acids or fewer), which may limit its use for longer sequences.


#Install Dependencies

##Install Conda

pyPept has been written in Python 3.9 or higher using standard Python libraries together with the following external packages:

- RDKit 2020 or later
- BioPython 1.7.9

Both packages can be installed using package managers such as Conda.

##install pyPept directly from GitHub.

In [ ]:
!pip install -q git+https://github.com/Boehringer-Ingelheim/pyPept.git

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.2/37.2 MB 19.0 MB/s eta 0:00:00


#Import Libraries

In [ ]:
# PyPept modules
from pyPept.sequence import Sequence
from pyPept.sequence import correct_pdb_atoms
from pyPept.molecule import Molecule
from pyPept.converter import Converter
from pyPept.conformer import Conformer
from pyPept.conformer import SecStructPredictor

# RDKit modules
from rdkit import Chem
from rdkit.Chem import Draw

#Define Peptide Input



##BILN format:
BILN (Biopolymer Linear Notation) is a simplified linear format used in pyPept for structural generation and downstream analysis
- ac → N-terminal acetylation  
- am → C-terminal amidation  
- Single-letter amino acid codes represent residues

In [ ]:
# Start the Sequence object
biln = "ac-D-T-H-F-E-I-A-am"
seq = Sequence(biln)
# Correct atom names in the sequence object
seq = correct_pdb_atoms(seq)

##HELM vs BILN conversion:

- HELM (Hierarchical Editing Language for Macromolecules) is a standard notation used in computational chemistry.
- BILN is a simplified linear representation used by pyPept.

In this notebook, we demonstrate conversion from HELM → BILN → Molecular structure



In [ ]:
# Call the converter to change from HELM to BILN
from pyPept.converter import Converter

helm = "PEPTIDE1{A.G.Q.A.A.K.E.F.I.A.A}| PEPTIDE2{G.L.E.E} $PEPTIDE1,PEPTIDE2,6:R3-4:R3$$$V2.0"
b = Converter(helm=helm)
biln = b.get_biln()
seq = Sequence(biln)
# Correct atom names in the sequence object
seq = correct_pdb_atoms(seq)

##FASTA to BILN Conversion (Custom Implementation):

FASTA is a widely used format for representing amino acid sequences using single-letter codes. However, it does not contain structural or chemical modification information required for molecular modeling in `pyPept`.

BILN is a simplified linear representation used by pyPept.

Since `pyPept (v1.0.0)` does not provide a native FASTA → BILN converter, we implement a simple custom conversion strategy in this notebook.


In [ ]:
# Call the converter to change from FASTA to BILN
from pyPept.sequence import Sequence

def fasta_to_biln(fasta):
    return "-".join(fasta.strip())

fasta = "MKWVTFISLLFLFSSAYSRG"

biln = fasta_to_biln(fasta)
seq = Sequence(biln)
# Correct atom names in the sequence object
seq = correct_pdb_atoms(seq)

#Monomer Inspection

In [ ]:
# Loop wit the included monomers
mm_list = seq.s_monomers
for i, monomer in enumerate(mm_list):
    mon = monomer['m_romol']

#Molecular Visualization

The peptide structure is visualized in both:

- 2D (RDKit depiction)
- 3D (conformational geometry)

These representations help in understanding molecular shape and spatial arrangement.

##Generate RDKit Molecule (2D representation)

In [ ]:
# Generate the RDKit object
mol = Molecule(seq)
romol = mol.get_molecule(fmt='ROMol')
print("The SMILES of the peptide is: {}".format(Chem.MolToSmiles(romol)))
Draw.MolToFile(romol, 'peptide.png', size=(1200, 1200))

##Generate 3D Conformer

In [ ]:
# Create the peptide conformer with corrected atom names and secondary structure
# Obtain peptide main chain to predict the secondary structure
fasta = Conformer.get_peptide(biln)
secstruct = SecStructPredictor.predict_active_ss(fasta)
# Generate the conformer
romol = Conformer.generate_conformer(romol, secstruct, generate_pdb=True)